# CN7 가스: RG3 공식 가스17개 보조 지도학습
정상 대비 상대값으로 제품 차이를 보정하고 보조학습을 시도한다. CN7만 학습한 모델과 구분한다.


In [1]:
from pathlib import Path
import nbformat
old=nbformat.read("notebooks/170_cn7_gas_persistent_relation_experts.ipynb",as_version=4)
oldcode="\n".join(c.source for c in old.cells if c.cell_type=="code")
exec(oldcode[:oldcode.index("def predict(parent,q):")])
exec(oldcode[oldcode.index("def stats(a,b):"):oldcode.index("records=[];choices=[]")])
BASIC=["Mold_Temperature_3","Mold_Temperature_4","Max_Injection_Speed","Filling_Time",
 "Max_Injection_Pressure","Max_Switch_Over_Pressure","Plasticizing_Position",
 "Plasticizing_Time","Cushion_Position","Average_Back_Pressure","Max_Screw_RPM"]
raw=d[[v+"_raw" for v in BASIC]].to_numpy(float)
donor=pd.read_csv(OUT/"original_labeled_shot_features.csv",parse_dates=["TimeStamp"])
donor=donor[donor["product"].eq("RG3")].copy()
donor["gas"]=donor.Reason.fillna("").str.contains("가스")
donor=donor[donor.gas|donor.PassOrFail.eq(0)].reset_index(drop=True)
assert len(donor)==583 and donor.gas.sum()==17
assert set(d.TimeStamp).isdisjoint(set(donor.TimeStamp))
donor_y=donor.gas.to_numpy(int)
donor_raw=donor[BASIC].to_numpy(float)
def norm(a,normal):
 imp=SimpleImputer(keep_empty_features=True).fit(a[normal]);aa=imp.transform(a)
 floor=[]
 for k in range(aa.shape[1]):
  vals=np.unique(np.round(aa[normal,k],4));dif=np.diff(vals);nz=dif[dif>1e-8]
  floor.append(max(float(np.min(nz)) if len(nz) else .01,1e-4))
 scale=np.maximum(np.std(aa[normal],axis=0),floor)
 return np.clip((aa-np.median(aa[normal],axis=0))/scale,-20,20)
donor_z=np.empty_like(donor_raw)
for run in donor.comparison_block.unique():
 ids=np.flatnonzero(donor.comparison_block.eq(run))
 donor_z[ids]=norm(donor_raw[ids],donor_y[ids]==0)
def target_z(ref):
 normal=ref[y[ref]==0];ans=np.empty_like(raw)
 for run in pd.unique(g):
  ni=normal[g[normal]==run]
  if len(ni)<8:ni=normal
  ids=np.r_[ni,np.flatnonzero(g==run)]
  zz=norm(raw[ids],np.arange(len(ids))<len(ni))
  ans[g==run]=zz[len(ni):]
 return ans
def embeddings(z):
 m3,m4,speed,fill,pressure,switch,meter,plastic,cushion,bp,rpm=z.T
 return {"flow6":np.column_stack([m3,speed,fill,pressure,switch,pressure-speed]),
 "load6":np.column_stack([meter,plastic,cushion,bp,rpm,meter-cushion]),
 "mechanism_joint8":np.column_stack([np.minimum(-m3,speed),np.minimum(-fill,speed),
  np.minimum(-pressure,speed),np.minimum(-m3,-fill),
  np.minimum(meter,plastic),np.minimum(cushion,plastic),
  np.minimum(bp,-rpm),np.abs(m4-m3)])}
dx=embeddings(donor_z)
VIEWS=list(dx);HEADS=["LR","RF","ET","mean3"]
memo={};audit=[]
def predict(parent,q):
 key=(tuple(np.flatnonzero(parent)),tuple(q))
 if key in memo:return memo[key]
 ctx=train_context(parent,q);ref=np.flatnonzero(ctx&eligible)
 assert not ctx[q].any() and set(ref).isdisjoint(q)
 audit.append({"timestamp_overlap":len(set(d.TimeStamp.iloc[ref])&set(d.TimeStamp.iloc[q])),
 "test_raw_in_training_history":int(ctx[q].sum()),"train_CN7_gas":int(y[ref].sum()),"donor_RG3_gas":17})
 tz=embeddings(target_z(ref));out={}
 yy=np.r_[y[ref],donor_y]
 # Fixed source weight; not selected from outer test labels.
 weights=np.r_[np.ones(len(ref)),np.full(len(donor),.5)]
 for v in VIEWS:
  a=np.vstack([tz[v][ref],dx[v]]);b=tz[v][q]
  models={"LR":LogisticRegression(C=.1,class_weight="balanced",max_iter=2000),
  "RF":RandomForestClassifier(n_estimators=40,max_depth=4,min_samples_leaf=3,class_weight="balanced",random_state=42,n_jobs=2),
  "ET":ExtraTreesClassifier(n_estimators=40,max_depth=4,min_samples_leaf=3,class_weight="balanced",random_state=42,n_jobs=2)}
  for h,m in models.items():out[v+"_"+h]=m.fit(a,yy,sample_weight=weights).predict_proba(b)[:,1]
  out[v+"_mean3"]=np.mean([out[v+"_"+h] for h in models],axis=0)
 memo[key]=(out,ref,ctx);return memo[key]
b=pd.read_csv(SOURCE/"cn7_165_gas_supervised_complementary_predictions.csv",parse_dates=["TimeStamp"]).set_index("TimeStamp").loc[d.TimeStamp]
base165=b.three_specialists_confirmed.to_numpy(int)
a=pd.read_csv(OUT/"cn7_170_persistence_predictions.csv",parse_dates=["TimeStamp"])
a=a[(a.model=="persistent_abs4_ET")&(a.rule=="normal95")].set_index("TimeStamp").loc[d.TimeStamp]
base170=base165|a.prediction.to_numpy(int)
a=pd.read_csv(OUT/"cn7_171_flow_predictions.csv",parse_dates=["TimeStamp"])
a=a[(a.model=="flow_signed6_LR")&(a.rule=="normal99")].set_index("TimeStamp").loc[d.TimeStamp]
base171=base170|a.prediction.to_numpy(int)
records=[];choices=[]
for held in pd.unique(blocks):
 q=np.flatnonzero(blocks==held);outer,ref,ctx=predict(np.ones(N,bool),q)
 inner={n:np.full(len(ref),np.nan) for n in outer}
 for ib in pd.unique(innerblocks[ref]):
  mask=innerblocks[ref]==ib;ip,_,_=predict(ctx,ref[mask])
  for n in inner:inner[n][mask]=ip[n]
 options=[]
 for name in outer:
  for quant in [.95,.97,.99,.995]:
   cut=float(np.nextafter(np.quantile(inner[name][y[ref]==0],quant,method="higher"),np.inf))
   met=stats(y[ref],inner[name]>=cut)
   if name.endswith("_mean3"):options.append((met["F1"],met["TP"],-met["FP"],name,quant,cut))
   for j,i in enumerate(q):records.append({"model":name,"rule":f"inner_normal{quant}",
    "TimeStamp":d.TimeStamp.iloc[i],"actual":int(y[i]),"eligible":int(eligible[i]),
    "score":float(outer[name][j]),"threshold":cut,"prediction":int(outer[name][j]>=cut)})
 best=max(options);_,_,_,name,quant,cut=best
 choices.append({"heldout_block":held,"model":name,"quantile":quant,"inner_F1":best[0]})
 for j,i in enumerate(q):records.append({"model":"inner_selected_mean3","rule":"inner_selection",
  "TimeStamp":d.TimeStamp.iloc[i],"actual":int(y[i]),"eligible":int(eligible[i]),
  "score":float(outer[name][j]),"threshold":cut,"prediction":int(outer[name][j]>=cut)})
 print(f"Finished RG3-supported {len(choices)}/20",flush=True)
 pd.DataFrame(choices).to_csv(OUT/"cn7_173_transfer_progress.csv",index=False)
pp=pd.DataFrame(records);rows=[]
for (name,rule),a in pp.groupby(["model","rule"],sort=False):
 a=a.set_index("TimeStamp").loc[d.TimeStamp];pr=a.prediction.to_numpy(int)
 for policy,base in [("standalone",np.zeros(N,int)),("add_to_165",base165),("add_to_170",base170),("add_to_171",base171)]:
  rows.append({"model":name,"rule":rule,"policy":policy,"N":605,"gas":13,**stats(y[eligible],(pr|base)[eligible])})
mm=pd.DataFrame(rows)
pp.to_csv(OUT/"cn7_173_transfer_predictions.csv",index=False,encoding="utf-8-sig")
mm.to_csv(OUT/"cn7_173_transfer_metrics.csv",index=False,encoding="utf-8-sig")
pd.DataFrame(audit).to_csv(OUT/"cn7_173_transfer_audit.csv",index=False)
assert pd.DataFrame(audit).timestamp_overlap.eq(0).all()
assert pd.DataFrame(audit).test_raw_in_training_history.eq(0).all()
top=mm.sort_values(["F1","FP"],ascending=[False,True]).head(15)
best=top.iloc[0]
a=pp[(pp.model==best.model)&(pp.rule==best.rule)].set_index("TimeStamp").loc[d.TimeStamp]
base={"standalone":np.zeros(N,int),"add_to_165":base165,"add_to_170":base170,"add_to_171":base171}[best.policy]
pr=a.prediction.to_numpy(int)|base
cases=d[["TimeStamp","gas","any_fail","run_id","seq"]].copy();cases["baseline"]=base;cases["candidate"]=pr
cases.to_csv(OUT/"cn7_173_transfer_best_cases.csv",index=False,encoding="utf-8-sig")
print("\nTOP\n",top.to_string(index=False))
print("\nINNER SELECTED\n",mm[mm.model.eq("inner_selected_mean3")].to_string(index=False))
print("\nGAS\n",cases[cases.gas].to_string(index=False))
lines=["# CN7 가스: RG3 가스17개를 보조 학습한 상대 공정 전문가","",
 "CN7 정상592/가스13의 동일 시간블록20/purge10/nested5. CN7 평가 샷은 학습에 포함하지 않는다.",
 "RG3 공식 정상566/가스17를 보조 학습으로 추가한다. 원시 절대값을 섞지 않고 각 제품/Run의 학습 정상 기준으로 표준화한다.",
 "CN7 라벨만 사용하는 모델과 다르며 RG3 보조 학습 사용을 반드시 표시한다. 19가스 데이터는 사용하지 않는다.",
 "보조 샘플 가중치는0.5 고정. RF/ET/LR 및 평균 앙상블.",
 "고정 최고는 이미 본 평가 결과에서 후보를 고른 개발 성적이다. 새 Run 독립검증이 아니다.","",
 "## 최고 고정 후보",top.to_string(index=False),"",
 "## 안쪽 선택",mm[mm.model.eq("inner_selected_mean3")].to_string(index=False),"",
 "## 가스13개",cases[cases.gas].to_string(index=False),"",
 "## 전체606샷 가스 판정",str(stats(y,pr))]
(ROOT/"reports/cn7_173_transfer_review.md").write_text("\n".join(lines),encoding="utf-8")
(OUT/"cn7_173_transfer_provenance.json").write_text(json.dumps({
 "input_sha256":hashlib.sha256(INPUT.read_bytes()).hexdigest(),"validation":"same20/purge10/nested5",
 "donor":"RG3 official normal566/gas17, no extra19 labels",
 "donor_weight":.5,"normalization":"per product/run reference normal only",
 "caveat":"test-informed development features and post-hoc best selection"},ensure_ascii=False,indent=2),encoding="utf-8")


Finished RG3-supported 1/20


Finished RG3-supported 2/20


Finished RG3-supported 3/20


Finished RG3-supported 4/20


Finished RG3-supported 5/20


...

Finished RG3-supported 18/20


Finished RG3-supported 19/20


Finished RG3-supported 20/20



TOP
                  model              rule     policy   N  gas  TP  FP  FN  TN  precision   recall       F1
   mechanism_joint8_LR  inner_normal0.99 add_to_170 605   13   8   5   5 587   0.615385 0.615385 0.615385
   mechanism_joint8_LR  inner_normal0.99 add_to_171 605   13   9   8   4 584   0.529412 0.692308 0.600000
   mechanism_joint8_LR inner_normal0.995 add_to_171 605   13   9   8   4 584   0.529412 0.692308 0.600000
mechanism_joint8_mean3 inner_normal0.995 add_to_171 605   13   9   8   4 584   0.529412 0.692308 0.600000
   mechanism_joint8_LR inner_normal0.995 add_to_170 605   13   7   4   6 588   0.636364 0.538462 0.583333
mechanism_joint8_mean3 inner_normal0.995 add_to_170 605   13   7   4   6 588   0.636364 0.538462 0.583333
           flow6_mean3 inner_normal0.995 add_to_171 605   13   9   9   4 583   0.500000 0.692308 0.580645
              load6_RF inner_normal0.995 add_to_171 605   13   9   9   4 583   0.500000 0.692308 0.580645
              load6_ET inner_normal0.995

353